[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c09_undo.ipynb)

# Leibniz's wrong guess: integration by substitution, parts and partial fractions

A computer finds the slope of e to the minus x squared at once. Ask it for the integral, and it gives up on a
formula and answers with a new name. Going forwards always works. Going backwards means recognising which slope
rule made the function, and sometimes no rule did.

The idea: **every method of integration is a slope rule, run backwards.** And you never have to trust an answer:
differentiate it, and see if the original comes back.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from top to bottom. When a slider appears, drag it; when the answer
box appears, type into it and press Enter. Nothing can break: if something looks odd, run everything again from
the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and the tools. Run this cell first. (In the browser, sympy loads the first time: give it a few seconds.)
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Text
import sys
if "pyodide" in sys.modules:                 # the browser version fetches sympy once
    import piplite
    await piplite.install(["sympy"])
import sympy as sp
from sympy.parsing.sympy_parser import (parse_expr, standard_transformations,
                                        implicit_multiplication_application, convert_xor)

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"   # the function · slopes · areas

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

x = sp.symbols("x")
TRANSFORMS = standard_transformations + (implicit_multiplication_application, convert_xor)

def read(s):
    """Turn what you type into maths: 2x cos(x^2), x*exp(x), ln(x), sqrt(x) all work."""
    return parse_expr(s, transformations=TRANSFORMS, local_dict={"x": x, "e": sp.E, "ln": sp.log})

def plain(f):
    """A formula the way you'd type it: 2·x·cos(x^2)."""
    return sp.sstr(f).replace("**", "^").replace("*", "·")

print("ready: the slope of e^(−x²) is", plain(sp.diff(sp.exp(-x**2), x)))

## 1 · Forwards is easy, backwards is not

Below is the curve y = e^(−x²) (teal). Drag the slider: the coral line is its slope at that point, which a slope
rule gives at once. The lilac area is the area under the curve from 0 up to the slider. That running area is a
perfectly real number at every point, but there is no formula for it made of the usual functions.

In [ ]:
def forwards_backwards(at=0.8):
    xs = np.linspace(-3, 3, 600)
    f = np.exp(-xs ** 2)
    y0 = math.exp(-at ** 2)
    m = -2 * at * y0                                   # the slope, from the chain rule
    fig, ax = plt.subplots(figsize=(10, 4.6))
    ax.plot(xs, f, color=TEAL, lw=3, label="y = e^(−x²)")
    lo, hi = sorted((0.0, at))
    sel = (xs >= lo) & (xs <= hi)
    ax.fill_between(xs[sel], 0, f[sel], color=LILAC, alpha=0.45, label="area from 0 to here")
    tx = np.linspace(at - 0.9, at + 0.9, 10)
    ax.plot(tx, y0 + m * (tx - at), color=CORAL, lw=2.4, label=f"slope here = {m:+.3f}")
    ax.plot([at], [y0], "o", color=CORAL, ms=9)
    area = math.sqrt(math.pi) / 2 * math.erf(at)
    ax.set_ylim(-0.3, 1.3); ax.set_xlabel("x"); ax.legend(loc="upper left")
    ax.set_title(f"slope: one rule, at once   ·   area from 0: {area:+.4f}, found by slicing")
    plt.show()

interact(forwards_backwards, at=FloatSlider(value=0.8, min=-2.5, max=2.5, step=0.05, description="x"));

**Notice.** The slope comes from one rule: −2x times e^(−x²). The area keeps growing as you drag right, and levels
off near 0.886, which is half of √π. Nobody has a formula for it. Liouville proved, in the 1830s, that none exists.

## 2 · Substitution: the chain rule, backwards

The chain rule says the slope of sin(x²) is cos(x²) × 2x: the outside's slope, times the inside's slope. So when an
integral shows an inside and its slope side by side, like 2x cos(x²), it was made by the chain rule. Call the inside
u = x²; the integral is sin(x²) + C.

Drag the slider. The teal curve is 2x cos(x²). The lilac curve is our answer, sin(x²). Its slope at the slider (the
coral line) always equals the height of the teal curve there.

In [ ]:
def substitution(at=1.0):
    xs = np.linspace(-2.5, 2.5, 800)
    fig, ax = plt.subplots(figsize=(10, 4.6))
    ax.plot(xs, 2 * xs * np.cos(xs ** 2), color=TEAL, lw=3, label="2x cos(x²)  (the integral's inside)")
    ax.plot(xs, np.sin(xs ** 2), color=LILAC, lw=2.4, label="sin(x²)  (the answer)")
    m = 2 * at * math.cos(at ** 2)
    y0 = math.sin(at ** 2)
    tx = np.linspace(at - 0.4, at + 0.4, 10)
    ax.plot(tx, y0 + m * (tx - at), color=CORAL, lw=2.4)
    ax.plot([at], [y0], "o", color=CORAL, ms=8); ax.plot([at], [m], "o", color=TEAL, ms=8)
    ax.set_ylim(-5.5, 5.5); ax.set_xlabel("x"); ax.legend(loc="lower left")
    ax.set_title(f"slope of sin(x²) at x = {at:.2f}: {m:+.3f}   ·   height of 2x cos(x²): {m:+.3f}")
    plt.show()

interact(substitution, at=FloatSlider(value=1.0, min=-2.4, max=2.4, step=0.05, description="x"));

**Notice.** The two numbers in the title are always the same. Now rub out the 2x: the integral of cos(x²) alone has
no formula made of the usual functions (its running area is called a Fresnel integral). A smaller formula can be a
much harder integral.

## 3 · Integration by parts: the product rule, rearranged

The product rule says the slope of u × v is u × (slope of v) + v × (slope of u). Rearranged and integrated:
**∫ u dv = uv − ∫ v du**. It helps when the leftover integral, ∫ v du, is easier than the one you started with.
For ∫ x eˣ dx, choose which part is u and see what is left over.

In [ ]:
CHOICES = {"u = x,  dv = eˣ dx": (x, sp.exp(x), "easier: no x left"),
           "u = eˣ,  dv = x dx": (sp.exp(x), x, "harder: now there's an x² in it")}

def by_parts(choice="u = x,  dv = eˣ dx"):
    u, dv, verdict = CHOICES[choice]
    v = sp.integrate(dv, x)
    left = sp.simplify(v * sp.diff(u, x))
    print(f"u = {u},   du = {sp.diff(u, x)} dx")
    print(f"dv = {dv} dx,   v = {v}")
    print(f"∫ x eˣ dx  =  {u * v}  −  ∫ {left} dx")
    print("the leftover integral is", verdict)
    print("the whole answer:", sp.integrate(x * sp.exp(x), x), "+ C")

interact(by_parts, choice=Dropdown(options=list(CHOICES), value="u = x,  dv = eˣ dx", description="choose"));

**Notice.** With u = x, the leftover is ∫ eˣ dx, which is easy. With u = eˣ, the leftover has x² in it: worse. Pick
u to be the part that gets simpler when you take its slope.

## 4 · Partial fractions, and Leibniz's wrong guess

Adding fractions forwards: 1/(x − 1) − 1/(x + 1) = 2/(x² − 1). Backwards, a fraction is split into simple pieces,
and each piece integrates to a log or an inverse tangent. The trick needs the bottom to break into real pieces.

In 1702 Leibniz claimed that x⁴ + a⁴ can never be split into two real quadratics. Drag a: the teal curve is
x⁴ + a⁴, and the dashed coral curve is the product (x² + √2·ax + a²)(x² − √2·ax + a²). They sit exactly on top of
each other, for every a.

In [ ]:
print("forwards:", sp.together(1 / (x - 1) - 1 / (x + 1)))
print("backwards:", sp.apart(1 / (x ** 2 - 1), x))
print("integral:", sp.integrate(1 / (x ** 2 - 1), x), "+ C")

def leibniz(a=1.0):
    xs = np.linspace(-2.2, 2.2, 600)
    r2 = math.sqrt(2)
    lhs = xs ** 4 + a ** 4
    rhs = (xs ** 2 + r2 * a * xs + a ** 2) * (xs ** 2 - r2 * a * xs + a ** 2)
    fig, ax = plt.subplots(figsize=(10, 4.4))
    ax.plot(xs, lhs, color=TEAL, lw=7, alpha=0.55, label=f"x⁴ + {a:g}⁴")
    ax.plot(xs, rhs, color=CORAL, lw=2, ls="--", label="(x² + √2·ax + a²)(x² − √2·ax + a²)")
    ax.set_ylim(-1, 25); ax.set_xlabel("x"); ax.legend(loc="upper center")
    ax.set_title(f"largest gap between them: {np.max(np.abs(lhs - rhs)):.1e}")
    plt.show()

interact(leibniz, a=FloatSlider(value=1.0, min=0.2, max=2.0, step=0.1, description="a"));

**Notice.** The curve never touches the axis, so it has no simple factors. Yet it still splits into two real
quadratics: x⁴ + 1 = (x² + 1)² − 2x², a square minus a square. Sympy agrees:

In [ ]:
print(sp.factor(x ** 4 + 1, extension=sp.sqrt(2)))

## 5 · The bell curve: a real area with no formula

The area under y = e^(−x²) from end to end is exactly √π ≈ 1.772. Drag the slider to cut it into more strips and
add them up: the total closes in on √π. The running area from 0 to x has its own name, the error function (named
by Glaisher in 1871): area = (√π ÷ 2) erf(x).

In [ ]:
def bell(strips=8):
    a, b = -4.0, 4.0
    w = (b - a) / strips
    mids = a + w * (np.arange(strips) + 0.5)
    total = float(np.sum(np.exp(-mids ** 2) * w))
    xs = np.linspace(a, b, 600)
    fig, ax = plt.subplots(figsize=(10, 4.4))
    ax.bar(mids, np.exp(-mids ** 2), width=w, color=LILAC, alpha=0.45, edgecolor=LILAC)
    ax.plot(xs, np.exp(-xs ** 2), color=TEAL, lw=3)
    ax.set_ylim(0, 1.15); ax.set_xlabel("x")
    ax.set_title(f"{strips} strips: total {total:.6f}   ·   √π = {math.sqrt(math.pi):.6f}")
    plt.show()

interact(bell, strips=IntSlider(value=8, min=2, max=60, step=1, description="strips"));

**Notice.** Even a dozen strips gets very close. That is how computers find erf: not by a formula, by slicing.

## 6 · Check any integral: differentiate it back

Pick an integral, type your answer in the box, and press **Enter**. The notebook finds the slope of your answer. If
you get the original back, you're right (the + C doesn't matter, since a constant has no slope). Write powers with
^ or **, and products with a space or *: `sin(x^2)`, `x*exp(x) - exp(x)`, `ln(x)`.

In [ ]:
INTEGRALS = {"∫ 2x cos(x²) dx": 2 * x * sp.cos(x ** 2), "∫ x eˣ dx": x * sp.exp(x),
             "∫ 1/(x² − 1) dx": 1 / (x ** 2 - 1), "∫ 2x(x² + 1)³ dx": 2 * x * (x ** 2 + 1) ** 3,
             "∫ x ln x dx": x * sp.log(x)}

def check_answer(integral="∫ 2x cos(x²) dx", answer=""):
    f = INTEGRALS[integral]
    print(f"{'the original:':<16}{plain(f)}")
    if not answer.strip():
        print("\ntype your answer in the box above and press Enter"); return
    try:
        F = read(answer)
    except Exception:
        print("\ncouldn't read that: try sin(x^2) or x*exp(x)"); return
    slope = sp.diff(F, x)
    print(f"{'your answer:':<16}{plain(F)}")
    print(f"{'its slope:':<16}{plain(sp.simplify(slope))}")
    right = sp.simplify(slope - f) == 0
    print("\n" + ("✓ the original comes back: you're right" if right else "✗ not the original: try again"))
    xs = np.linspace(0.15, 2.2, 400)
    fn, sn = sp.lambdify(x, f, "numpy"), sp.lambdify(x, slope, "numpy")
    with np.errstate(all="ignore"):
        fy, sy = np.broadcast_to(fn(xs), xs.shape).astype(float), np.broadcast_to(sn(xs), xs.shape).astype(float)
    fy[np.abs(fy) > 60] = np.nan; sy[np.abs(sy) > 60] = np.nan
    fig, ax = plt.subplots(figsize=(10, 3.6), layout="constrained")
    ax.plot(xs, fy, color=TEAL, lw=7, alpha=0.55, label="the original")
    ax.plot(xs, sy, color=CORAL if right else DIM, lw=2, ls="--", label="the slope of your answer")
    ax.set_xlabel("x"); ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
    ax.set_title("the dashes sit on the thick line" if right else "the dashes miss the thick line")
    plt.show()

interact(check_answer, integral=Dropdown(options=list(INTEGRALS), value="∫ 2x cos(x²) dx", description="integral"),
         answer=Text(value="", placeholder="your answer, e.g. sin(x^2)", description="answer", continuous_update=False));

**Notice.** Try a wrong answer too, like `cos(x^2)`: the dashes leave the thick line. Then try the challenge,
∫ x ln x dx (hint: by parts, with u = ln x).

## 7 · The exam problem: change the limits too

Find the integral from 0 to 1 of 2x(x² + 1)³ dx. Let u = x² + 1, so du = 2x dx. The limits must change too: when
x = 0, u = 1; when x = 1, u = 2. So it is the integral from 1 to 2 of u³ du = [u⁴ ÷ 4] from 1 to 2 = 4 − ¼ = 15/4.
The working that loses a mark keeps the old limits, 0 and 1, and gets ¼.

In [ ]:
u = sp.symbols("u")
right = sp.integrate(u ** 3, (u, 1, 2))
slip = sp.integrate(u ** 3, (u, 0, 1))
direct = sp.integrate(2 * x * (x ** 2 + 1) ** 3, (x, 0, 1))
print("new limits, 1 to 2:", right, "   old limits, 0 to 1 (the slip):", slip, "   directly in x:", direct)
assert right == direct == sp.Rational(15, 4) and slip == sp.Rational(1, 4)

**Try it.**
1. ∫ 3x²(x³ + 2)⁴ dx. (Look for the inside and its slope.)
2. ∫ x cos x dx. (Which part should be u?)
3. The integral from 0 to 2 of x e^(x²) dx. (Change the limits.)

<details><summary>Answers</summary>

1. u = x³ + 2, du = 3x² dx: (x³ + 2)⁵ ÷ 5 + C.
2. u = x, dv = cos x dx: x sin x + cos x + C.
3. u = x², limits 0 to 4: ½ (e⁴ − 1) ≈ 26.80.

</details>

Check 1 and 2 in the answer box above (pick any integral, or change `INTEGRALS` and run the cell again).

In [ ]:
print(sp.integrate(3 * x ** 2 * (x ** 3 + 2) ** 4, x), "|", sp.integrate(x * sp.cos(x), x), "|",
      sp.integrate(x * sp.exp(x ** 2), (x, 0, 2)), "≈", round(float(sp.integrate(x * sp.exp(x ** 2), (x, 0, 2))), 2))

## Sources
MacTutor History of Mathematics (St Andrews): "The fundamental theorem of algebra" (Leibniz 1702; Euler 1742) and
"Joseph Liouville"; H. K. Strick, *Joseph Liouville* (MacTutor); Wikipedia, *Partial fraction decomposition*,
*Nonelementary integral*, *Error function* (Glaisher, 1871), *Risch algorithm* (1968). All retrieved 2026-10-24.